# NLP HW6: BERT-base and RoBERTa-base grids for Section 3.1.4 (Colab GPU)

GenAI disclosure: Claude (Anthropic, model claude-opus-5-5) wrote this notebook. Brian W. Onieal directed the work and ran it.

This runs the Section 3.1.4 full fine-tuning grids (learning rates 1e-4, 5e-4, 1e-3 times 7 and 9 epochs, batch 32, seed 42) with the same `base_classification.py` as the repo. The first batch is the six RoBERTa-base runs. The second batch, added at the end while the first was running, is the six BERT-base runs plus two RoBERTa-base checks outside the grid. Both batches ran on an NVIDIA L4 in bfloat16 (`--amp`). Each run's console log, plot, and parsed accuracies go to `My Drive/NLP-HW6-colab/` as soon as the run ends, so a disconnect loses at most the run in progress. Running the notebook again skips finished runs.

**To run:** Runtime > Change runtime type > a GPU (an L4 took 5 to 7 minutes per run; a T4 has no bfloat16, so the first code cell switches to float16 with loss scaling). Then Runtime > Run all, and allow Google Drive access when asked. Keep this tab open until the last cell prints ALL DONE.

In [ ]:
import torch, sys, platform
print(platform.python_version(), torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again"
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
# bf16 needs compute capability 8.0+ (Ampere or newer); a T4 is 7.5, so it uses fp16 with loss scaling
AMP_ARGS = ["--amp"] if CAP >= (8, 0) else ["--amp", "--amp_dtype", "fp16"]
print(GPU, CAP, AMP_ARGS)

In [ ]:
# Library versions close to the repo's pins that install cleanly on Colab's Python
!pip -q install "transformers==4.46.3" "datasets==2.21.0" "evaluate==0.4.3"
import transformers, datasets, evaluate
print(transformers.__version__, datasets.__version__, evaluate.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/NLP-HW6-colab'
os.makedirs(OUT, exist_ok=True)
print(OUT)

In [ ]:
%%writefile base_classification.py
# GenAI disclosure: Claude (Anthropic, model claude-opus-5-5) completed the TODOs in this
# file (evaluate_model and the training loop in train) and added the --seed, --amp, and
# --amp_dtype arguments marked "Added" below. Everything else is the course skeleton as
# shipped. I directed the work, ran the experiments through Claude on my own GPU and on a
# Google Colab GPU, and reviewed the code.
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm import tqdm
import evaluate as evaluate
from transformers import get_scheduler
from transformers import AutoModelForSequenceClassification
import argparse
import subprocess
import matplotlib.pyplot as plt

# Added: mixed precision, switched on with --amp. Off by default, so a plain run is the
# skeleton's float32 training. --amp_dtype fp16 is for GPUs without bfloat16 (e.g. a T4) and
# uses loss scaling.
AMP = False
AMP_DTYPE = torch.bfloat16
SCALER = None

def print_gpu_memory():
    """
    Print the amount of GPU memory used by the current process
    This is useful for debugging memory issues on the GPU
    """
    # check if gpu is available
    if torch.cuda.is_available():
        print("torch.cuda.memory_allocated: %fGB" % (torch.cuda.memory_allocated(0) / 1024 / 1024 / 1024))
        print("torch.cuda.memory_reserved: %fGB" % (torch.cuda.memory_reserved(0) / 1024 / 1024 / 1024))
        print("torch.cuda.max_memory_reserved: %fGB" % (torch.cuda.max_memory_reserved(0) / 1024 / 1024 / 1024))

        p = subprocess.check_output('nvidia-smi')
        print(p.decode("utf-8"))


class BoolQADataset(torch.utils.data.Dataset):
    """
    Dataset for the dataset of BoolQ questions and answers
    """

    def __init__(self, passages, questions, answers, tokenizer, max_len):
        self.passages = passages
        self.questions = questions
        self.answers = answers
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.answers)

    def __getitem__(self, index):
        """
        This function is called by the DataLoader to get an instance of the data
        :param index:
        :return:
        """

        passage = str(self.passages[index])
        question = self.questions[index]
        answer = self.answers[index]

        # this is input encoding for your model. Note, question comes first since we are doing question answering
        # and we don't wnt it to be truncated if the passage is too long
        input_encoding = question + " [SEP] " + passage

        # encode_plus will encode the input and return a dictionary of tensors
        encoded_review = self.tokenizer.encode_plus(
            input_encoding,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )

        return {
            'input_ids': encoded_review['input_ids'][0],  # we only have one example in the batch
            'attention_mask': encoded_review['attention_mask'][0],
            # attention mask tells the model where tokens are padding
            'labels': torch.tensor(answer, dtype=torch.long)  # labels are the answers (yes/no)
        }


def evaluate_model(model, dataloader, device):
    """ Evaluate a PyTorch Model
    :param torch.nn.Module model: the model to be evaluated
    :param torch.utils.data.DataLoader test_dataloader: DataLoader containing testing examples
    :param torch.device device: the device that we'll be training on
    :return accuracy
    """

    # load metrics
    dev_accuracy = evaluate.load('accuracy')

    # turn model into evaluation mode
    model.eval()

    # iterate over the dataloader
    for batch in dataloader:
        # TODO: implement the evaluation function
        # raise NotImplementedError("You need to implement the evaluation function")
        # get the input_ids, attention_mask from the batch and put them on the device
        # Hints:
        # - see the getitem function in the BoolQADataset class for how to access the input_ids and attention_mask
        # - use to() to move the tensors to the device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # forward pass
        # name the output as `output`
        # no_grad: evaluation needs no gradients, and skipping them saves GPU memory
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP):  # autocast added for --amp
            output = model(input_ids=input_ids, attention_mask=attention_mask)

        # your code ends here

        predictions = output.logits
        predictions = torch.argmax(predictions, dim=1)
        dev_accuracy.add_batch(predictions=predictions, references=batch['labels'])

    # compute and return metrics
    return dev_accuracy.compute()


def train(mymodel, num_epochs, train_dataloader, validation_dataloader, test_dataloder, device, lr, small_subset=False):
    """ Train a PyTorch Module

    :param torch.nn.Module mymodel: the model to be trained
    :param int num_epochs: number of epochs to train for
    :param torch.utils.data.DataLoader train_dataloader: DataLoader containing training examples
    :param torch.utils.data.DataLoader validation_dataloader: DataLoader containing validation examples
    :param torch.device device: the device that we'll be training on
    :param float lr: learning rate
    :return None
    """

    # here, we use the AdamW optimizer. Use torch.optim.Adam.
    # instantiate it on the untrained model parameters with a learning rate of 5e-5
    print(" >>>>>>>>  Initializing optimizer")
    optimizer = torch.optim.AdamW(mymodel.parameters(), lr=lr)

    # now, we set up the learning rate scheduler
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=50,
        num_training_steps=len(train_dataloader) * num_epochs
    )

    loss = torch.nn.CrossEntropyLoss()
    
    epoch_list = []
    train_acc_list = []
    dev_acc_list = []

    for epoch in range(num_epochs):

        # put the model in training mode (important that this is done each epoch,
        # since we put the model into eval mode during validation)
        mymodel.train()

        # load metrics
        train_accuracy = evaluate.load('accuracy')

        print(f"Epoch {epoch + 1} training:")

        for i, batch in tqdm(enumerate(train_dataloader)):

            """
            You need to make some changes here to make this function work.
            Specifically, you need to: 
            Extract the input_ids, attention_mask, and labels from the batch; then send them to the device. 
            Then, pass the input_ids and attention_mask to the model to get the logits.
            Then, compute the loss using the logits and the labels.
            Then, call loss.backward() to compute the gradients.
            Then, call optimizer.step()  to update the model parameters.
            Then, call lr_scheduler.step() to update the learning rate.
            Then, call optimizer.zero_grad() to reset the gradients for the next iteration.
            """

            # TODO: implement the training loop
            # raise NotImplementedError("You need to implement this function")

            # get the input_ids, attention_mask, and labels from the batch and put them on the device
            # Hints: similar to the evaluate_model function
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # forward pass
            # name the output as `output`
            # Hints: refer to the evaluate_model function on how to get the predictions (logits)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=AMP):  # Added: --amp
                output = mymodel(input_ids=input_ids, attention_mask=attention_mask)
                predictions = output.logits

                # compute the loss using the loss function
                # (named model_loss so it doesn't overwrite `loss`, the CrossEntropyLoss object)
                model_loss = loss(predictions, labels)

            # loss backward
            if SCALER is not None:  # Added: fp16 needs loss scaling
                SCALER.scale(model_loss).backward()
            else:
                model_loss.backward()

            # update the model parameters with optimizer and lr_scheduler step
            if SCALER is not None:  # Added: unscales, skips the step if gradients overflowed
                SCALER.step(optimizer)
                SCALER.update()
            else:
                optimizer.step()
            lr_scheduler.step()

            # zero the gradients
            optimizer.zero_grad()

            # your code ends here

            predictions = torch.argmax(predictions, dim=1)

            # update metrics
            train_accuracy.add_batch(predictions=predictions, references=batch['labels'])
            
        # print evaluation metrics
        print(f" ===> Epoch {epoch + 1}")
        train_acc = train_accuracy.compute()
        print(f" - Average training metrics: accuracy={train_acc}")
        train_acc_list.append(train_acc['accuracy'])

        # normally, validation would be more useful when training for many epochs
        val_accuracy = evaluate_model(mymodel, validation_dataloader, device)
        print(f" - Average validation metrics: accuracy={val_accuracy}")
        dev_acc_list.append(val_accuracy['accuracy'])
        
        epoch_list.append(epoch)
        
    # generate plots here
    plt.clf()
    plt.plot(epoch_list, train_acc_list, 'b', label='train')
    if not small_subset:
        plt.plot(epoch_list, dev_acc_list, 'g', label='valid')
    plt.xlabel('Training Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    save_path = "overfit.png" if small_subset else "base_full.png"
    plt.savefig(save_path)

def pre_process(model_name, batch_size, device, small_subset, seed=None):
    # download dataset
    print("Loading the dataset ...")
    dataset = load_dataset("boolq")
    # Added: a fixed seed, so every run in the hyper-parameter search sees the same
    # train/test split. The skeleton's unseeded shuffle drew a new split on every run.
    dataset = dataset.shuffle(seed=seed)  # shuffle the data

    print("Slicing the data...")
    if small_subset:
        # use this tiny subset for debugging the implementation
        dataset_train_subset = dataset['train'][:10]
        dataset_dev_subset = dataset['train'][:10]
        dataset_test_subset = dataset['train'][:10]
    else:
        # since the dataset does not come with any validation data,
        # split the training data into "train" and "dev"
        dataset_train_subset = dataset['train'][:8000]
        dataset_dev_subset = dataset['validation']
        dataset_test_subset = dataset['train'][8000:]

    print("Size of the loaded dataset:")
    print(f" - train: {len(dataset_train_subset['passage'])}")
    print(f" - dev: {len(dataset_dev_subset['passage'])}")
    print(f" - test: {len(dataset_test_subset['passage'])}")

    # maximum length of the input; any input longer than this will be truncated
    # we had to do some pre-processing on the data to figure what is the length of most instances in the dataset
    max_len = 128

    print("Loading the tokenizer...")
    mytokenizer = AutoTokenizer.from_pretrained(model_name)

    print("Loding the data into DS...")
    train_dataset = BoolQADataset(
        passages=list(dataset_train_subset['passage']),
        questions=list(dataset_train_subset['question']),
        answers=list(dataset_train_subset['answer']),
        tokenizer=mytokenizer,
        max_len=max_len
    )
    validation_dataset = BoolQADataset(
        passages=list(dataset_dev_subset['passage']),
        questions=list(dataset_dev_subset['question']),
        answers=list(dataset_dev_subset['answer']),
        tokenizer=mytokenizer,
        max_len=max_len
    )
    test_dataset = BoolQADataset(
        passages=list(dataset_test_subset['passage']),
        questions=list(dataset_test_subset['question']),
        answers=list(dataset_test_subset['answer']),
        tokenizer=mytokenizer,
        max_len=max_len
    )

    print(" >>>>>>>> Initializing the data loaders ... ")
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
    validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

    # from Hugging Face (transformers), read their documentation to do this.
    print("Loading the model ...")
    pretrained_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    print("Moving model to device ..." + str(device))
    pretrained_model.to(device)
    return pretrained_model, train_dataloader, validation_dataloader, test_dataloader


# the entry point of the program
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--small_subset", action='store_true',
                        help="When set true, only run training on a small subset of the data, used for 3.1.1")
    parser.add_argument("--num_epochs", type=int, default=1)
    parser.add_argument("--lr", type=float, default=5e-5)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--device", type=str, default="cuda")
    parser.add_argument("--model", type=str, default="distilbert-base-uncased")
    # Added: seed for the data shuffle and for torch (classifier init, dropout)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--amp", action="store_true", help="mixed precision (Added)")
    parser.add_argument("--amp_dtype", choices=["bf16", "fp16"], default="bf16")  # Added

    args = parser.parse_args()
    print(f"Specified arguments: {args}")

    assert type(args.small_subset) == bool, "small_subset must be a boolean"
    torch.manual_seed(args.seed)  # Added
    AMP = args.amp  # Added: module-level flag read by train() and evaluate_model()
    if AMP and args.amp_dtype == "fp16":  # Added
        AMP_DTYPE = torch.float16
        SCALER = torch.cuda.amp.GradScaler()

    # load the data and models
    pretrained_model, train_dataloader, validation_dataloader, test_dataloader = pre_process(args.model,
                                                                                             args.batch_size,
                                                                                             args.device,
                                                                                             args.small_subset,
                                                                                             args.seed)
    print(" >>>>>>>>  Starting training ... ")
    train(pretrained_model, args.num_epochs, train_dataloader, validation_dataloader, test_dataloader, args.device, args.lr, args.small_subset)
    
    # print the GPU memory usage just to make sure things are alright
    print_gpu_memory()

    val_accuracy = evaluate_model(pretrained_model, validation_dataloader, args.device)
    print(f" - Average DEV metrics: accuracy={val_accuracy}")

    test_accuracy = evaluate_model(pretrained_model, test_dataloader, args.device)
    print(f" - Average TEST metrics: accuracy={test_accuracy}")


In [ ]:
import json, re, shutil, subprocess, time

MODEL, BS = "roberta-base", 32
RUNS = [(lr, ep) for lr in (1e-4, 5e-4, 1e-3) for ep in (7, 9)]
PAT = {"dev": r"Average DEV metrics: accuracy=\{'accuracy': ([0-9.]+)\}",
       "test": r"Average TEST metrics: accuracy=\{'accuracy': ([0-9.]+)\}",
       "train_acc": r"Average training metrics: accuracy=\{'accuracy': ([0-9.]+)\}",
       "dev_acc": r"Average validation metrics: accuracy=\{'accuracy': ([0-9.]+)\}"}
res_path = f"{OUT}/results_colab.json"
results = json.load(open(res_path)) if os.path.exists(res_path) else {}

for lr, ep in RUNS:
    name = f"3.1.4_grid_roberta-base_lr{lr:g}_ep{ep}"
    log = f"{OUT}/{name}.log"
    if os.path.exists(log) and "Average TEST metrics" in open(log, encoding="utf-8", errors="replace").read():
        print("skip", name, "(already done)")
        continue
    cmd = [sys.executable, "base_classification.py", "--device", "cuda", "--model", MODEL,
           "--batch_size", str(BS), "--lr", str(lr), "--num_epochs", str(ep)] + AMP_ARGS
    print("run ", name, " ".join(cmd[1:]), flush=True)
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as out, open(f"/content/{name}.err", "w") as err:
        out.write(f"# Google Colab, {GPU}, torch {torch.__version__}, transformers {transformers.__version__}, datasets {datasets.__version__}\n")
        out.write("$ python " + " ".join(cmd[1:]) + "\n")
        out.flush()
        rc = subprocess.call(cmd, stdout=out, stderr=err,
                             env=dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1"))
    text = open(log, encoding="utf-8", errors="replace").read()
    if rc != 0:  # keep the end of stderr so a crash can be diagnosed
        shutil.copy(f"/content/{name}.err", f"{OUT}/{name}.err")
    if os.path.exists("base_full.png"):
        shutil.move("base_full.png", f"{OUT}/{name}.png")
    r = {"model": MODEL, "batch_size": BS, "lr": lr, "epochs": ep, "amp_args": AMP_ARGS, "gpu": GPU,
         "returncode": rc, "wall_seconds": round(time.time() - t0, 1)}
    for k, p in PAT.items():
        vals = [float(x) for x in re.findall(p, text)]
        r[k] = vals if k in ("train_acc", "dev_acc") else (vals[-1] if vals else None)
    results[name] = r
    json.dump(results, open(res_path, "w"), indent=2)
    print(f"     rc={rc} dev={r['dev']} test={r['test']} in {r['wall_seconds']/60:.1f} min", flush=True)

print("ALL DONE")

In [ ]:
for k, r in results.items():
    print(f"{k:45s} dev {r['dev']}  test {r['test']}  {r['wall_seconds']/60:.1f} min")

## Second batch: the BERT-base grid and two RoBERTa-base checks

Added while the first batch was running, once it was clear the BERT-base grid would also run faster here than on the laptop, and once the first RoBERTa-base run had collapsed to always-yes. This is a cleaned-up copy of the cell that ran. The runs, settings, and commands match the logs in Drive, which record each command on their second line. Same script, seed, split, and batch 32. Two runs outside the grid follow the BERT-base grid:

* `3.1.4_control_roberta-base_fp32_lr0.0001_ep7`: the first grid run again without `--amp`, to see whether bfloat16 caused the collapse
* `3.1.4_extra_roberta-base_lr2e-05_ep9`: RoBERTa-base at 2e-5, inside the 1e-5 to 3e-5 range the RoBERTa paper used for fine-tuning

In [ ]:
RUNS2 = [(f"3.1.4_grid_bert-base-uncased_lr{lr:g}_ep{ep}", "bert-base-uncased", lr, ep, AMP_ARGS)
         for lr in (1e-4, 5e-4, 1e-3) for ep in (7, 9)]
RUNS2 += [("3.1.4_control_roberta-base_fp32_lr0.0001_ep7", "roberta-base", 1e-4, 7, []),
          ("3.1.4_extra_roberta-base_lr2e-05_ep9", "roberta-base", 2e-5, 9, AMP_ARGS)]
res2_path = f"{OUT}/results_colab2.json"
results2 = json.load(open(res2_path)) if os.path.exists(res2_path) else {}

for name, model, lr, ep, amp_args in RUNS2:
    log = f"{OUT}/{name}.log"
    if os.path.exists(log) and "Average TEST metrics" in open(log, encoding="utf-8", errors="replace").read():
        print("skip", name, "(already done)")
        continue
    cmd = [sys.executable, "base_classification.py", "--device", "cuda", "--model", model,
           "--batch_size", str(BS), "--lr", str(lr), "--num_epochs", str(ep)] + amp_args
    print("run ", name, " ".join(cmd[1:]), flush=True)
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as out, open(f"/content/{name}.err", "w") as err:
        out.write(f"# Google Colab, {GPU}, torch {torch.__version__}, transformers {transformers.__version__}, datasets {datasets.__version__}\n")
        out.write("$ python " + " ".join(cmd[1:]) + "\n")
        out.flush()
        rc = subprocess.call(cmd, stdout=out, stderr=err,
                             env=dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1"))
    text = open(log, encoding="utf-8", errors="replace").read()
    if rc != 0:
        shutil.copy(f"/content/{name}.err", f"{OUT}/{name}.err")
    if os.path.exists("base_full.png"):
        shutil.move("base_full.png", f"{OUT}/{name}.png")
    r = {"model": model, "batch_size": BS, "lr": lr, "epochs": ep, "amp_args": amp_args, "gpu": GPU,
         "returncode": rc, "wall_seconds": round(time.time() - t0, 1)}
    for k, p in PAT.items():
        vals = [float(x) for x in re.findall(p, text)]
        r[k] = vals if k in ("train_acc", "dev_acc") else (vals[-1] if vals else None)
    results2[name] = r
    json.dump(results2, open(res2_path, "w"), indent=2)
    print(f"     rc={rc} dev={r['dev']} test={r['test']} in {r['wall_seconds']/60:.1f} min", flush=True)

print("ALL DONE")